In [ ]:
import pandas as pd
from time import time
import InsurAutoML
# from InsurAutoML import InformedAutoTabular
from InsurAutoML.hpo.informed.fixed import InformedAutoTabular

seed = 42
n_trials = 1024
N_ESTIMATORS = 4
TIMEOUT = (n_trials / 4) * 450

InsurAutoML.set_seed(seed)

In [ ]:
import openml
from sklearn.model_selection import train_test_split
# Get dataset by ID
dataset = openml.datasets.get_dataset(42727)
# Get the data itself
data, _, _, _ = dataset.get_data()
# train/test split
# first time running
train, test = train_test_split(
    data, test_size=0.1, random_state=seed
)
pd.DataFrame(train).to_csv("train.csv", index=False)
pd.DataFrame(test).to_csv("test.csv", index=False)

# # Use the same train/test split across all models for 2+ runs
# train, test = pd.read_csv("train.csv"), pd.read_csv("test.csv")

In [ ]:
response = "percent_pell_grant"
features = [col for col in train.columns if col != response]
train_X, test_X, train_y, test_y = (
    train[features],
    test[features],
    train[response],
    test[response],
)

In [ ]:
# fit AutoML model
mol = InformedAutoTabular(
    model_name="college_informed_{}".format(n_trials),
    max_evals=n_trials,
    n_estimators=N_ESTIMATORS,
    timeout=TIMEOUT,
    models = ["LightGBM_Regressor", "XGBoost_Regressor", "ExtraTreesRegressor"],
    search_algo="Optuna",
    objective="MSE",
    cpu_threads=12,
    seed=seed,
)
start_time = time()
mol.fit(train_X, train_y)
end_time = time()

In [ ]:
from sklearn.metrics import root_mean_squared_error
train_pred = mol.predict(train_X)
test_pred = mol.predict(test_X)
print("Number of trials: ", n_trials)
print("Time taken (s): ", end_time - start_time)
print("Train RMSE: ", root_mean_squared_error(train_y, train_pred))
print("Test RMSE: ", root_mean_squared_error(test_y, test_pred))